# 02 — The 313 Sieve and the 49999 Un-Sieve

**Fourth Age Paper companion notebook.** `ScalarContextPropagation` — background
for **C2/C3** (the domain the box kite lives on is the factoring map over
`ℕ`; this notebook is where that domain's own two boundaries come from).

Two counter-rotating trees over the composites `≤ N` (`N = 10⁵` here),
built with **The Two Trees engine** (`GenerationalLineage/engine/lineage.py`,
`un_sieve`; doc `ValaQuenta/wiki/un_sieve.md`):

- **Telperion's book** (the sieve): a composite dies the moment its
  *smallest* prime factor is struck. Extinction — bounded, front-loaded.
- **Laurelin's book** (the un-sieve): from "just prime numbers," turn
  primes on one at a time; a composite is *born* the moment its *last*
  needed prime factor switches on. Construction — paid for in full.

Provenance: **ESTABLISHED** — Sieve of Eratosthenes (classical); the
recursive un-sieve and the two named boundaries are this project's own
reading (`un_sieve.md`), computed here, not asserted.

In [1]:
import sys, os, math
sys.path.insert(0, os.path.expanduser("~/Projects/ThePlace/GenerationalLineage/engine"))
import lineage

N = 100_000
r = lineage.un_sieve(N)
print(f"N = {N:,}")
print(f"primes <= N:      {r['n_primes']:,}")
print(f"composites <= N:  {r['n_composites']:,}")


N = 100,000
primes <= N:      9,592
composites <= N:  90,407


## Boundary 1 — extinction completes at 313

The standard optimised sieve only needs to strike multiples of `p`
starting at `p²` (every smaller multiple of `p` was already struck by a
smaller prime factor). So `p` contributes a **new** strike only while
`p² ≤ N`. The largest such prime, for `N = 10⁵`, is the "313 Sieve"
boundary — verified directly, not looked up:

In [2]:
def sieve_primes(n):
    flags = [True] * (n + 1)
    flags[0] = flags[1] = False
    for i in range(2, int(n ** 0.5) + 1):
        if flags[i]:
            for j in range(i * i, n + 1, i):
                flags[j] = False
    return [i for i, f in enumerate(flags) if f]

primes = sieve_primes(N)
ext_b_direct = max(p for p in primes if p * p <= N)
print(f"largest prime p with p^2 <= {N}: {ext_b_direct}")
print(f"  313^2 = {313**2:,}  (<= {N:,}: {313**2 <= N})")
print(f"  317^2 = {317**2:,}  (<= {N:,}: {317**2 <= N})   -- 317 is the next prime after 313")
assert ext_b_direct == 313 == r['extinction_boundary_prime']
print("MATCH: extinction_boundary_prime = 313, confirmed by direct sieve trace and by un_sieve().")


largest prime p with p^2 <= 100000: 313
  313^2 = 97,969  (<= 100,000: True)
  317^2 = 100,489  (<= 100,000: False)   -- 317 is the next prime after 313
MATCH: extinction_boundary_prime = 313, confirmed by direct sieve trace and by un_sieve().


## Boundary 2 — birth doesn't finish until 49999

Laurelin's book waits for a composite's *largest* prime factor. The
smallest composite with greatest-prime-factor `p` is `2p` (its only other
factor is the smallest possible, 2), so `p` can only father a composite
inside the domain while `2p ≤ N`. The largest such prime, for `N = 10⁵`,
is the birth boundary:

In [3]:
birth_b_direct = max(p for p in primes if 2 * p <= N)
print(f"largest prime p with 2p <= {N}: {birth_b_direct}")
print(f"  2 * 49999 = {2*49999:,}  (<= {N:,}: {2*49999 <= N})")
print(f"  2 * 50021 = {2*50021:,}  (<= {N:,}: {2*50021 <= N})   -- 50021 is the next prime after 49999")
assert birth_b_direct == 49999 == r['birth_boundary_prime']
print("MATCH: birth_boundary_prime = 49999, confirmed by direct trace and by un_sieve().")


largest prime p with 2p <= 100000: 49999
  2 * 49999 = 99,998  (<= 100,000: True)
  2 * 50021 = 100,042  (<= 100,000: False)   -- 50021 is the next prime after 49999
MATCH: birth_boundary_prime = 49999, confirmed by direct trace and by un_sieve().


## The split — exclusion is free, construction is paid

- **Telperion (extinction, low→high spf):** entropy 2.491 bits, done at
  `√N` = prime 313.
- **Laurelin (birth, low→high gpf):** entropy 9.685 bits, not finished
  until `N/2` = prime 49999.

`D == reverse(A)` exactly (birth high→low is extinction low→high, read
backwards, bit for bit) — the trees are mirrors of each other when keyed
on the same prime factor. The one non-mirror comparison, **C against A**
(birth by *greatest* factor vs. death by *smallest*), is where the real
cost shows up: the same information spread across ~5,000 generations
instead of ~65.

In [4]:
print("orders (generation range, entropy, top-pass share):")
for k, s in r['orders'].items():
    print(f"  {k:22s}  range={s['gen_range']}  H={s['entropy_bits']:.3f} bits  "
          f"top_pass_share={s['top_pass_share']:.4f}")

print()
print("D == reverse(A) exactly:", r['D_equals_reverse_A'])
print(f"H(C) - H(A) = {r['H_C_minus_H_A']:.5f} bits")

_spf, _gpf, _ = lineage._spf_gpf_tables(N)
born_after_313 = sum(1 for n in range(4, N + 1)
                      if _spf[n] != n and _gpf[n] > 313)
print(f"\nfraction of composites <= {N:,} born strictly after the extinction "
      f"boundary (313): {born_after_313 / r['n_composites']:.4f}")


orders (generation range, entropy, top-pass share):
  A_extinction_lo_hi      range=(0, 64)  H=2.491 bits  top_pass_share=0.5530
  B_extinction_hi_lo      range=(4459, 9591)  H=9.685 bits  top_pass_share=0.0000
  C_birth_lo_hi           range=(0, 5132)  H=9.685 bits  top_pass_share=0.0002
  D_birth_hi_lo           range=(9527, 9591)  H=2.491 bits  top_pass_share=0.0000

D == reverse(A) exactly: True
H(C) - H(A) = 7.19355 bits

fraction of composites <= 100,000 born strictly after the extinction boundary (313): 0.6045


## Clocked by zeta — the gap is invariant, but one clock narrows it

Re-running the same reads with the ordinal prime rank replaced by
zeta-derived orders (`ln p/√p`, Riemann–Siegel `θ`, `Z`-sign, spiral
phase) leaves `H(C) − H(A)` invariant to five decimals — a combinatorial
invariant of `ℕ`, not an artefact of counting order. The one clock that
*does* move it: birth ordered by the real Riemann zeros `γ_k` instead of
the integers narrows the gap by ≈15% (`+4.30 → +3.66` bits at `N=8000`).
That computation lives in
`RiemannHypothesisProof/ADDENDUM_recursive_unsieve_2026-08-30.md §B.1–D.1`
— referenced here, not reproduced (it needs the zero-finder from notebook
01 run at scale, which is exactly today's separate testing pass).

## Summary

| quantity | value | verified here |
|---|---|---|
| extinction boundary (largest `p`, `p² ≤ N`) | **313** | direct sieve trace + `un_sieve()`, agree |
| birth boundary (largest `p`, `2p ≤ N`) | **49999** | direct trace + `un_sieve()`, agree |
| `D == reverse(A)` | `True` | `un_sieve()` |
| `H(C) − H(A))` | ≈ **7.1936** bits | `un_sieve()` |

Both boundary primes are exactly what `un_sieve()` reports, and both are
reproduced independently by the plain-arithmetic argument above (`p²≤N`
and `2p≤N`) — not merely quoted from `ValaQuenta/wiki/un_sieve.md`.